# Using the Best Association Rule Model for Recommendations

In [1]:
# Using the Best Association Rule Model for Recommendations
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


## 1. Load Best Association Rules

We load the rules selected in the model selection step.  
Each rule has:

- `antecedents` (items on the left side)  
- `consequents` (items to recommend)  
- `support`, `confidence`, `lift`, etc.[web:80][web:252]


In [2]:
# ============================================================
# 1. Load Best Association Rules
# ============================================================

BEST_DIR = "/kaggle/input/04-best-model-selection/best_association_model"

BEST_RULES_FILE = os.path.join(BEST_DIR, "best_model_rules_apriori.csv")

rules = pd.read_csv(BEST_RULES_FILE)

print("Best rules shape:", rules.shape)
display(rules.head())


Best rules shape: (54, 21)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,...,jaccard,certainty,kulczynski,antecedent_len,consequent_len,rule_len,experiment_id,min_support,min_confidence,min_lift
0,frozenset({'Frozen Organic Wild Blueberries'}),frozenset({'Bag of Organic Bananas'}),0.01010,0.12335,0.00305,0.301980,2.448157,1.0,0.001804,1.255910,...,0.023390,0.203765,0.163353,1,1,2,376748ce,0.002,0.3,1.0
1,"frozenset({""Organic D'Anjou Pears""})",frozenset({'Bag of Organic Bananas'}),0.01580,0.12335,0.00505,0.319620,2.591165,1.0,0.003101,1.288472,...,0.037658,0.223887,0.180280,1,1,2,376748ce,0.002,0.3,1.0
2,frozenset({'Organic Gala Apples'}),frozenset({'Bag of Organic Bananas'}),0.02385,0.12335,0.00730,0.306080,2.481392,1.0,0.004358,1.263329,...,0.052180,0.208441,0.182630,1,1,2,376748ce,0.002,0.3,1.0
3,frozenset({'Organic Navel Orange'}),frozenset({'Bag of Organic Bananas'}),0.01450,0.12335,0.00515,0.355172,2.879387,1.0,0.003361,1.359511,...,0.038809,0.264441,0.198462,1,1,2,376748ce,0.002,0.3,1.0
4,frozenset({'Organic Snipped Green Beans'}),frozenset({'Bag of Organic Bananas'}),0.00545,0.12335,0.00200,0.366972,2.975050,1.0,0.001328,1.384853,...,0.015773,0.277902,0.191593,1,1,2,376748ce,0.002,0.3,1.0


## 2. Prepare Rules for Recommendation

We need to convert the `antecedents` and `consequents` columns from their string representation back into Python sets, so we can check if a user basket contains a rule’s antecedent.[web:262][web:266]


In [3]:
# ============================================================
# 2. Parse antecedents and consequents into Python sets
# ============================================================

import ast

def parse_itemset(s):
    """
    Convert a string representation of a Python set or frozenset
    like "frozenset({'Banana', 'Milk'})" to a real Python set.
    """
    if pd.isna(s):
        return set()
    # Handle formats like "frozenset({'x', 'y'})"
    s = str(s)
    if s.startswith("frozenset("):
        inner = s[len("frozenset("):-1]  # strip wrapper
        parsed = ast.literal_eval(inner)
    else:
        parsed = ast.literal_eval(s)
    return set(parsed)

rules = rules.copy()
rules["antecedents_set"] = rules["antecedents"].apply(parse_itemset)
rules["consequents_set"] = rules["consequents"].apply(parse_itemset)

display(rules[["antecedents", "antecedents_set", "consequents", "consequents_set"]].head())


,antecedents,antecedents_set,consequents,consequents_set
0,frozenset({'Frozen Organic Wild Blueberries'}),{Frozen Organic Wild Blueberries},frozenset({'Bag of Organic Bananas'}),{Bag of Organic Bananas}
1,"frozenset({""Organic D'Anjou Pears""})",{Organic D'Anjou Pears},frozenset({'Bag of Organic Bananas'}),{Bag of Organic Bananas}
2,frozenset({'Organic Gala Apples'}),{Organic Gala Apples},frozenset({'Bag of Organic Bananas'}),{Bag of Organic Bananas}
3,frozenset({'Organic Navel Orange'}),{Organic Navel Orange},frozenset({'Bag of Organic Bananas'}),{Bag of Organic Bananas}
4,frozenset({'Organic Snipped Green Beans'}),{Organic Snipped Green Beans},frozenset({'Bag of Organic Bananas'}),{Bag of Organic Bananas}


## 3. Rule-Based Recommendation Function

We implement a simple recommendation function:

- Input: list of products currently in the user’s basket.  
- For each rule where **antecedent is a subset of the basket**, we collect the consequents.  
- We score recommendations by aggregating metrics (e.g. confidence × lift) and sort them.[web:262][web:265][web:268]


In [4]:
# ============================================================
# 3. Recommendation Function
# ============================================================

def recommend_from_rules(
    user_basket,
    rules_df,
    top_k=5,
    min_confidence=0.0,
    min_lift=1.0
):
    """
    Recommend products based on association rules and the user's basket.
    
    Parameters
    ----------
    user_basket : list of str
        Products currently in the basket.
    rules_df : pd.DataFrame
        DataFrame of rules with 'antecedents_set', 'consequents_set',
        'confidence', and 'lift'.
    top_k : int
        Maximum number of recommended products.
    min_confidence : float
        Minimum confidence threshold to consider a rule.
    min_lift : float
        Minimum lift threshold to consider a rule.
    
    Returns
    -------
    recommendations : list of (product, score, source_rule_idx)
    """
    basket_set = set(user_basket)
    candidates = []

    for idx, row in rules_df.iterrows():
        antecedent = row["antecedents_set"]
        consequent = row["consequents_set"]
        conf = row.get("confidence", 0.0)
        lift = row.get("lift", 1.0)

        # Filter by thresholds
        if conf < min_confidence or lift < min_lift:
            continue

        # Check if rule applies: antecedent ⊆ basket
        if antecedent.issubset(basket_set):
            # products to recommend are consequents not already in basket
            for item in consequent:
                if item not in basket_set:
                    score = conf * lift  # simple combined score
                    candidates.append((item, score, idx))

    if not candidates:
        return []

    # Aggregate scores per product (in case multiple rules suggest same item)
    rec_df = pd.DataFrame(candidates, columns=["product", "score", "rule_idx"])
    agg = rec_df.groupby("product").agg(
        total_score=("score", "sum"),
        best_score=("score", "max"),
        n_rules=("score", "count")
    ).reset_index()

    # Sort by total_score (or best_score) descending
    agg_sorted = agg.sort_values("total_score", ascending=False)

    # Take top-k and return with simple structure
    recommendations = []
    for _, row in agg_sorted.head(top_k).iterrows():
        recommendations.append(
            (row["product"], row["total_score"], row["n_rules"])
        )

    return recommendations


## 4. Handling Cases with No Applicable Rules

If no rule’s antecedent is a subset of the basket, the function returns an empty list.  
In a real system, you could:

- Fallback to popular products.  
- Use collaborative filtering or content-based methods.  
- Show category-level recommendations instead.[web:271][web:269]


In [5]:
# ============================================================
# 4. Demonstration with Example Baskets
# ============================================================

def print_recommendations(basket, recs):
    print(f"\nUser basket: {basket}")
    if not recs:
        print("No rule-based recommendations. (Fallback to other strategies.)")
    else:
        print("Recommended products (product, score, #supporting_rules):")
        for prod, score, n_rules in recs:
            print(f"  - {prod}  | score={score:.3f}, from {n_rules} rules")


# Example baskets (replace with real product names from your dataset)
example_baskets = [
    ["Banana", "Organic Strawberries", "Organic Whole Milk"],
    ["Bag of Organic Bananas", "Organic Baby Spinach"],
    ["Sparkling Water", "Lime", "Ice Cream"],
    ["Toilet Paper", "Dish Soap"],  # possibly no strong rules
]

for basket in example_baskets:
    recs = recommend_from_rules(
        user_basket=basket,
        rules_df=rules,
        top_k=5,
        min_confidence=0.1,
        min_lift=1.0
    )
    print_recommendations(basket, recs)



User basket: ['Banana', 'Organic Strawberries', 'Organic Whole Milk']
No rule-based recommendations. (Fallback to other strategies.)

User basket: ['Bag of Organic Bananas', 'Organic Baby Spinach']
No rule-based recommendations. (Fallback to other strategies.)

User basket: ['Sparkling Water', 'Lime', 'Ice Cream']
No rule-based recommendations. (Fallback to other strategies.)

User basket: ['Toilet Paper', 'Dish Soap']
No rule-based recommendations. (Fallback to other strategies.)


## 5. How This Can Be Used in Practice

### E-commerce Recommendation Systems

- **Cart-based suggestions**: For each active basket, compute rule-based recommendations and display them as “Frequently Bought Together” or “Customers Also Bought”.[web:269][web:276]  
- **Next-best offer**: When a user adds an item to the cart, recompute recommendations using rules whose antecedent contains that item.

### Cross-Selling Strategies

- Use high-lift rules (e.g., `Lift > 2`) to identify product pairs or bundles that co-occur much more than chance.  
- Design **product bundles** or **checkout add-ons** (“add milk to your cereal order”) based on these rules.[web:248][web:163]

### Marketing Campaigns

- Segment rules by department/aisle (e.g., produce, beverages) to design targeted promotions.  
- Use rules like `A, B → C` to create **email campaigns**: customers who recently bought A and B get a coupon for C.[web:194][web:271]  

In all cases, association rules act as a **transparent, interpretable layer**: each recommendation can be explained as “because you bought X and Y, we suggest Z”, which is valuable for user trust and business stakeholders.[web:274][web:277]
